In [12]:
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [13]:
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [14]:
# Configuration
MODEL_NAME = "UDPipe"
SAMPLE_TYPES = ["glosses"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("04-udpipe.ipynb")

In [15]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

In [16]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [22]:
def udpipe_processor(sample_texts, sample_type, pipeline):
    processed_results = []
    for sample_id, text in sample_texts:

        processed = pipeline.process(text)
        sentences = processed.strip().split('\n\n')

        for sent in sentences:
            lines = sent.split("\n")
            for idx, line in enumerate(lines):
                if line.startswith('#') or not line.strip():
                    continue

                fields = line.split('\t')
                if len(fields) != 10:
                    continue

                form = fields[1]
                lemma = fields[2]
                upos = fields[3]

                if sample_type == "glosses":
                    word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{idx}"
                else:
                    word_id = str(idx)

                processed_results.append({
                    "sample_id": sample_id,
                    "word_id": word_id,
                    "word": form,
                    "lemma": lemma,
                    "pos": upos
                })

    return processed_results


In [18]:
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [19]:
# Load the UDPipe model
model_path = "/Users/Thea/Desktop/LatinNLPTools/scripts/latin-ittb-ud-2.5-191206.udpipe"
model = Model.load(model_path)
if not model:
    raise Exception("Model not loaded!")

In [20]:
# Create a processing pipeline
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

In [23]:
sample_type = "glosses"
print(f"Processing {sample_type}...")

# Load gold standard data
gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

gold_df = pd.read_csv(gold_file)

# Extract text to reconstruct from words
sample_texts = word_joiner(gold_df)

# Process samples and measure time
start_time = time.time()

processed_results = udpipe_processor(sample_texts, sample_type, pipeline)

processing_time = time.time() - start_time
results["processing_times"][sample_type] = processing_time

print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

# Merge gold_df with processed_results
pred_df = pd.DataFrame(processed_results)

merged_df = pd.merge(gold_df, pred_df, on=['sample_id','word_id','word'], suffixes=('_gold', '_pred'))




Processing glosses...
Data processes with UDPipe in 0.15546298027038574 seconds.


In [24]:
#print(f"pred_df: {pred_df}")
print(f"merged_df: {merged_df}")

merged_df:    sample_id                                            word_id      word  \
0  BCr.30b36  http://gams.uni-graz.at/o:glossvibe.bvi#BCr.30...    mensis   
1  Sg.68.06a  http://gams.uni-graz.at/o:glossvibe.bvi#Sg.68....      dies   
2  Sg.68.06a  http://gams.uni-graz.at/o:glossvibe.bvi#Sg.68....    mensis   
3  BCr.32c12  http://gams.uni-graz.at/o:glossvibe.bvi#BCr.32...  kalendis   
4  BCr.32c12  http://gams.uni-graz.at/o:glossvibe.bvi#BCr.32...  ianuaris   

  lemma_gold pos_gold lemma_pred pos_pred  
0     mensis     NOUN      mensa     NOUN  
1       dies     NOUN       dies     NOUN  
2     mensis     NOUN      mensa     NOUN  
3    kalenda     NOUN   kalendus     VERB  
4  ianuarius     NOUN   ianuaris      ADJ  
